# 제14장: 도구 및 플랫폼

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch14.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### 거버넌스 도구와 플랫폼

## 모델 레지스트리와 카탈로그

### 모델 레지스트리의 필요성

### 주요 솔루션 비교

**Integrated Model Registry and Governance Management Class**

In [ ]:
class ModelRegistry:
    """Central registry for managing model lifecycles
    and governance policies"""
    def __init__(self):
        self.models = {}
        self.audit_log = []

    def register_model(self, model_id, metadata,
                       artifact_path):
        """Register a new model with mandatory metadata"""
        required = ['owner', 'risk_level',
                    'training_data', 'performance_metrics']
        for field in required:
            if field not in metadata:
                raise ValueError(
                    f"Missing required field: {field}")
        self.models[model_id] = {
            'metadata': metadata,
            'artifact_path': artifact_path,
            'stage': 'Development',
            'created_at': datetime.now()
        }
        self._log_audit('REGISTER', model_id)

    def request_promotion(self, model_id, target_stage,
                          requester):
        """Stage promotion with governance checks"""
        meta = self.models[model_id]['metadata']
        if meta['risk_level'] == RiskLevel.HIGH:
            return {"status": "pending_approval",
                    "approvers": ["CISO", "GovBoard"]}
        elif meta['risk_level'] == RiskLevel.MEDIUM:
            return {"status": "pending_approval",
                    "approvers": ["TeamLead"]}
        return {"status": "auto_approved"}

    def _log_audit(self, action, model_id):
        """Immutable audit log for registry operations"""
        self.audit_log.append({
            'timestamp': datetime.now(),
            'action': action, 'model_id': model_id,
            'hash': self._compute_log_hash()})

### 메타데이터 스키마 설계

**Comprehensive Model Metadata Schema (JSON)**

In [ ]:
MODEL_METADATA_SCHEMA = {
  # 1. Basic Identification
  "model_id": "mdl-2026-0142",
  "model_name": "credit_scoring_v3",
  "version": "3.2.1",
  "owner": {"team": "Risk Analytics",
            "lead": "kimjs@company.com"},
  "stage": "Production",
  # 2. Model Card (per Mitchell et al. 2019)
  "model_card": {
    "description": "Consumer credit scoring model",
    "intended_use": ["Credit assessment"],
    "out_of_scope_use": ["Employment screening"],
    "ethical_considerations": [
      "Age group performance disparity"]},
  # 3. Training Information
  "training": {
    "dataset_id": "ds-2025-089",
    "dataset_size": "2.4M records",
    "sensitive_features": ["age_group", "gender"],
    "algorithm": "XGBoost",
    "framework": "xgboost==2.0.3",
    "hyperparameters": {"max_depth": 8,
                        "learning_rate": 0.05}},
  # 4. Performance Metrics
  "metrics": {
    "accuracy": {"value": 0.891, "threshold": 0.85},
    "auc_roc": {"value": 0.934, "threshold": 0.90},
    "fairness": {"demographic_parity": 0.92,
                 "disparate_impact": 0.95}},
  # 5. Risk and Compliance
  "risk_classification": {
    "level": "HIGH",
    "ai_act_category": "High-Risk (Annex III)",
    "korea_ai_act": "High-Impact"},
  "compliance": {
    "status": "APPROVED",
    "regulations": [
      {"name": "AI Basic Act", "status": "compliant"},
      {"name": "EU AI Act", "status": "compliant"}],
    "approvals": [
      {"approver": "CISO", "date": "2026-01-25"},
      {"approver": "ModelRiskCommittee",
       "date": "2026-01-28"}]},
  # 6. Lineage Information
  "lineage": {
    "parent_model": "mdl-2026-0098",
    "data_sources": ["s3://data-lake/credit/raw/"],
    "feature_store_ref": "fs://credit_features_v2",
    "dependent_services": ["credit-api-v2"]}
}

### 조직 맞춤 구축 vs 상용 도입

## 실험 추적과 재현성 도구

### 재현성의 중요성

### 실험 추적 도구 비교

**MLflow Experiment Tracking with Governance Metadata**

In [ ]:
import mlflow
import hashlib
import subprocess
from datetime import datetime

class GovernanceAwareTracker:
    """MLflow-based experiment tracker with governance metadata"""

    def __init__(self, experiment_name, risk_level="MEDIUM"):
        self.experiment_name = experiment_name
        self.risk_level = risk_level
        mlflow.set_experiment(experiment_name)

    def start_governed_run(self, run_name, owner, data_version,
                           purpose, regulations=None):
        """Start an experiment run with governance metadata"""
        with mlflow.start_run(run_name=run_name) as run:
            # Governance metadata
            mlflow.set_tag("governance.owner", owner)
            mlflow.set_tag("governance.risk_level", self.risk_level)
            mlflow.set_tag("governance.purpose", purpose)
            mlflow.set_tag("governance.data_version", data_version)
            mlflow.set_tag("governance.regulations",
                           ",".join(regulations or []))
            mlflow.set_tag("governance.timestamp",
                           datetime.utcnow().isoformat())

            # Reproducibility metadata
            git_hash = subprocess.check_output(
                ["git", "rev-parse", "HEAD"]
            ).decode().strip()
            mlflow.set_tag("reproducibility.git_commit", git_hash)
            mlflow.set_tag("reproducibility.python_version",
                           sys.version)

            # Environment hash for reproducibility verification
            env_hash = self._compute_env_hash()
            mlflow.set_tag("reproducibility.env_hash", env_hash)

            return run

    def log_approval(self, run_id, approver, decision, comments=""):
        """Record model approval decision for audit trail"""
        with mlflow.start_run(run_id=run_id):
            mlflow.set_tag(f"approval.{approver}.decision", decision)
            mlflow.set_tag(f"approval.{approver}.timestamp",
                           datetime.utcnow().isoformat())
            mlflow.set_tag(f"approval.{approver}.comments", comments)

    def _compute_env_hash(self):
        """Compute hash of current environment for verification"""
        pip_freeze = subprocess.check_output(
            ["pip", "freeze"]
        ).decode()
        return hashlib.sha256(pip_freeze.encode()).hexdigest()[:12]

# Usage example
tracker = GovernanceAwareTracker(
    experiment_name="credit_scoring_v2",
    risk_level="HIGH"
)
run = tracker.start_governed_run(
    run_name="baseline_xgboost",
    owner="ml-team-lead@company.com",
    data_version="v2.3.1",
    purpose="Credit default prediction model update",
    regulations=["AI_Basic_Act", "Credit_Information_Act"]
)

### 대용량 데이터 버전 관리

**DVC-based Data Version Management with Governance Metadata**

In [ ]:
import subprocess, json, os
from datetime import datetime

class GovernanceDataVersionManager:
    """DVC-based data versioning with governance audit trail"""

    def __init__(self, project_root):
        self.project_root = project_root
        self.audit_log = os.path.join(
            project_root, ".governance", "data_audit.jsonl")

    def version_dataset(self, data_path, description, owner,
                        classification="INTERNAL"):
        """Version a dataset with DVC and record governance metadata"""
        subprocess.run(["dvc", "add", data_path], check=True)

        audit_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "data_path": data_path,
            "description": description,
            "owner": owner,
            "classification": classification,
            "dvc_hash": self._get_dvc_hash(data_path),
        }
        self._append_audit_log(audit_entry)

        # Commit DVC metadata and audit log to Git
        dvc_file = f"{data_path}.dvc"
        subprocess.run(
            ["git", "add", dvc_file, self.audit_log], check=True)
        subprocess.run(
            ["git", "commit", "-m",
             f"data: version {os.path.basename(data_path)} "
             f"- {description}"], check=True)
        subprocess.run(["dvc", "push"], check=True)
        return audit_entry

    def get_data_lineage(self, model_version):
        """Retrieve complete data lineage for a model version"""
        lineage = []
        with open(self.audit_log, "r") as f:
            for line in f:
                entry = json.loads(line)
                if entry.get("model_version") == model_version:
                    lineage.append(entry)
        return lineage

### 재현성 보장을 위한 환경 관리

## 모니터링 및 관측성(Observability) 플랫폼

### ML 모니터링의 특수성: Silent Failure

**Comprehensive Drift Detection with Multiple Statistical Tests**

In [ ]:
import numpy as np
from scipy import stats
from typing import Dict

class DriftDetector:
    """Multi-method drift detection for ML monitoring"""
    def __init__(self, significance_level: float = 0.05):
        self.significance_level = significance_level

    def ks_test(self, ref: np.ndarray,
                cur: np.ndarray) -> Dict:
        stat, p_val = stats.ks_2samp(ref, cur)
        return {"method": "KS-test", "statistic": stat,
                "p_value": p_val,
                "drift_detected": p_val < self.significance_level}

    def calculate_psi(self, ref: np.ndarray,
                      cur: np.ndarray, n_bins=10) -> Dict:
        bins = np.linspace(min(ref.min(), cur.min()),
                           max(ref.max(), cur.max()), n_bins+1)
        ref_hist, _ = np.histogram(ref, bins=bins)
        cur_hist, _ = np.histogram(cur, bins=bins)
        ref_pct = (ref_hist + 1e-6) / len(ref)
        cur_pct = (cur_hist + 1e-6) / len(cur)
        psi = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
        severity = ("none" if psi < 0.1
                     else "moderate" if psi < 0.25
                     else "significant")
        return {"method": "PSI", "psi_value": psi,
                "severity": severity, "drift_detected": psi >= 0.1}

    def js_divergence(self, ref: np.ndarray,
                      cur: np.ndarray, n_bins=50) -> Dict:
        bins = np.linspace(min(ref.min(), cur.min()),
                           max(ref.max(), cur.max()), n_bins+1)
        r, _ = np.histogram(ref, bins=bins, density=True)
        c, _ = np.histogram(cur, bins=bins, density=True)
        r_p, c_p = r/(r.sum()+1e-10), c/(c.sum()+1e-10)
        m = 0.5 * (r_p + c_p)
        js = 0.5*(stats.entropy(r_p,m)+stats.entropy(c_p,m))
        return {"method": "JS", "js_value": js,
                "drift_detected": js > 0.1}

    def run_all_tests(self, ref, cur) -> Dict:
        results = {"ks": self.ks_test(ref, cur),
                   "psi": self.calculate_psi(ref, cur),
                   "js": self.js_divergence(ref, cur)}
        n = sum(1 for r in results.values()
                if r["drift_detected"])
        results["consensus"] = {"overall_drift": n >= 2}
        return results

### 모니터링 플랫폼 비교

### 관측성(Observability) 아키텍처 설계

### GenAI 모니터링의 새로운 과제

**LLM Response Quality Monitor with Cost Tracking**

In [ ]:
import time
from dataclasses import dataclass, field
from typing import List

@dataclass
class LLMInferenceLog:
    prompt: str
    response: str
    model_id: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    timestamp: float = field(default_factory=time.time)

class LLMMonitor:
    """Monitor LLM quality, cost, and safety metrics"""
    def __init__(self, cost_per_1k_input: float = 0.01,
                 cost_per_1k_output: float = 0.03):
        self.cost_in = cost_per_1k_input
        self.cost_out = cost_per_1k_output
        self.logs: List[LLMInferenceLog] = []

    def log_inference(self, log: LLMInferenceLog) -> dict:
        self.logs.append(log)
        alerts = []
        cost = (log.input_tokens * self.cost_in / 1000
                + log.output_tokens * self.cost_out / 1000)
        if cost > self._cost_threshold():
            alerts.append({"type": "cost_anomaly",
                           "value": cost})

        hall_score = self._check_hallucination(
            log.prompt, log.response)
        if hall_score > 0.5:
            alerts.append({"type": "hallucination_risk",
                           "score": hall_score})

        safety = self._check_safety(log.response)
        if not safety["is_safe"]:
            alerts.append({"type": "safety_violation",
                           "details": safety["violations"]})

        if log.latency_ms > 5000:
            alerts.append({"type": "latency_sla_breach",
                           "latency_ms": log.latency_ms})
        return {"cost": cost, "alerts": alerts}

    def _check_hallucination(self, prompt, resp) -> float:
        # Production: NLI model or KB retrieval
        return 0.0

    def _check_safety(self, response: str) -> dict:
        # Production: moderation API + PII detection
        return {"is_safe": True, "violations": []}

    def _cost_threshold(self) -> float:
        if len(self.logs) < 100:
            return 0.10
        import numpy as np
        costs = [(l.input_tokens*self.cost_in/1000
                  + l.output_tokens*self.cost_out/1000)
                 for l in self.logs[-100:]]
        return np.mean(costs) + 2*np.std(costs)

    def get_daily_report(self) -> dict:
        import numpy as np
        if not self.logs: return {}
        total = sum(l.input_tokens*self.cost_in/1000
                    + l.output_tokens*self.cost_out/1000
                    for l in self.logs)
        lats = [l.latency_ms for l in self.logs]
        return {"total_requests": len(self.logs),
                "total_cost_usd": round(total, 2),
                "p95_latency_ms": round(
                    np.percentile(lats, 95), 1)}

## 컴플라이언스 자동화 도구

### 정책-as-코드(Policy-as-Code) 패러다임

**OPA 통합 정책-as-코드 엔진 구현**

In [ ]:
class GovernancePolicyEngine:
    """OPA(Open Policy Agent) 통합 AI 거버넌스 정책 엔진"""
    def __init__(self, opa_url="http://localhost:8181"):
        self.opa_url = opa_url
        self.policies = {"min_accuracy": 0.85,
                         "max_disparate_impact": 0.20,
                         "required_docs": ["model_card", "data_sheet"]}

    def evaluate_model(self, metadata):
        """모델 메타데이터에 대해 전체 거버넌스 정책 검증"""
        results = [self._check_fairness(metadata),
                   self._check_performance(metadata)]
        results.extend(self._query_opa(metadata))  # OPA 위임
        return results

    def _check_fairness(self, metadata):
        di = metadata.get("metrics", {}).get("disparate_impact", 0)
        if di > self.policies["max_disparate_impact"]:
            return PolicyCheckResult("fairness", "FAIL",
                f"DI ratio {di:.3f} exceeds threshold")
        return PolicyCheckResult("fairness", "PASS", "OK")

    def _query_opa(self, metadata):
        resp = requests.post(
            f"{self.opa_url}/v1/data/ai_governance/deploy",
            json={"input": metadata})
        return self._parse_opa_decision(resp.json())

### 자동화된 모델 검증 게이트

**CI/CD 거버넌스 게이트 구현**

In [ ]:
class GovernanceGate:
    """ML CI/CD 파이프라인의 거버넌스 게이트 관리자"""
    GATE_CONFIGS = {
        "post_training": {"checks": ["accuracy", "data_integrity"],
                          "auto_approve": True},
        "pre_staging":   {"checks": ["fairness_di", "model_card"],
                          "auto_approve": lambda m: m.risk != "HIGH"},
        "pre_production": {"checks": ["security_scan", "compliance"],
                           "auto_approve": lambda m: m.risk == "LOW"},
    }

    def execute_gate(self, gate_name, model_metadata):
        config = self.GATE_CONFIGS[gate_name]
        results = [self.policy_engine.evaluate(c, model_metadata)
                   for c in config["checks"]]
        if not all(r.status == "PASS" for r in results):
            return GateDecision(approved=False, results=results)
        auto_fn = config["auto_approve"]
        if callable(auto_fn) and not auto_fn(model_metadata):
            return GateDecision(approved=False,
                                reason="Manual approval required")
        return GateDecision(approved=True, results=results)

### 규정 준수 보고 자동화

**규정 준수 보고서 자동 생성기**

In [ ]:
class ComplianceReportGenerator:
    """규제 요구사항에 맞춘 자동 보고서 생성기"""
    TEMPLATES = {
        "eu_ai_act": ["system_description", "risk_classification",
                      "conformity_assessment", "post_market_monitoring"],
        "korean_ai_basic_law": ["system_overview", "risk_level",
                                "impact_assessment", "transparency_report"],
    }
    def generate_report(self, model_id, regulation):
        sections = self.TEMPLATES[regulation]
        data = {s: self._collect_data(model_id, s) for s in sections}
        report = self._render_template(regulation, data)
        report.metadata = {"generated_at": datetime.utcnow().isoformat(),
                           "model_id": model_id,
                           "data_sources": self._get_lineage(model_id)}
        self._archive_report(report)
        return report

### 통합 거버넌스 플랫폼의 부상

### 제14장 요약